In [1]:
import pandas as pd;
import numpy as np;
from SpatialCheck import isSpatialWithinBounds


# DOWNLOAD FILE FIRST: https://syd1.digitaloceanspaces.com/duckgoesmeow/bushfire-data/data.csv
FILE_PATH = "~/Desktop/archive/data.csv"
bushfire_df = pd.read_csv(FILE_PATH, low_memory=False)

columns_of_interest = ['OBJECTID','DISCOVERY_DATE', 'DISCOVERY_TIME', 'NWCG_GENERAL_CAUSE', 'CONT_DATE', 'CONT_TIME', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE' , 'LONGITUDE' , 'STATE' ]
renamed_columns = ["object_id", "discovery_date", "discovery_time", "general_cause", "controlled_date", "controlled_time", "fire_size", "fire_class", "latitude", "longitude", "state", "county"]
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Undefined',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}



# Let's rename the columns:
bushfire_df = bushfire_df.rename(columns={
                         'OBJECTID':'object_id',
                         'DISCOVERY_DATE':'discovery_date',
                         'DISCOVERY_TIME' : 'discovery_time',
                         'NWCG_GENERAL_CAUSE':'general_cause',
                         'CONT_DATE':'controlled_date',
                         'CONT_TIME':'controlled_time',
                         'FIRE_SIZE' : 'fire_size',
                         'FIRE_SIZE_CLASS' : 'fire_class',
                         'LATITUDE': 'latitude',
                         'LONGITUDE':'longitude',
                         'COUNTY' : 'county',
                         'STATE':'state'}).rename_axis('index_id')

In [2]:
bushfire_df = bushfire_df[renamed_columns]

In [3]:
bushfire_df['discovery_date'] = pd.to_datetime(arg=bushfire_df['discovery_date']).astype(str)

In [4]:
bushfire_df['discovery_time'] = pd.to_numeric(bushfire_df['discovery_time'], errors='coerce').fillna(0).astype(int).astype(str)

In [5]:
def format_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).zfill(4)
    if time_str == '0000' or time_str == '2400':
        return '00:00'
    elif 0 <= int(time_str) <= 2359:
        return f"{time_str[:2]}:{time_str[2:]}"
    else: 
        return np.nan
    

bushfire_df['discovery_time'] = bushfire_df['discovery_time'].apply(format_time)

In [6]:
bushfire_df['discovery_datetime'] = bushfire_df['discovery_date'] + " " + bushfire_df['discovery_time']

In [7]:
bushfire_df['discovery_datetime'] = pd.to_datetime(arg=bushfire_df['discovery_datetime'], format='%Y-%m-%d %H:%M')

In [8]:
bushfire_df.insert(11, 'discovery_day', bushfire_df['discovery_datetime'].dt.day_name())

In [9]:
bushfire_df['discovery_datetime'] = bushfire_df['discovery_datetime'].dt.strftime('%Y-%m-%d %H:%M')

In [10]:
bushfire_df.drop(columns=['discovery_date', 'discovery_time'], axis=1, inplace=True)

In [11]:
bushfire_df.drop(columns=['county'], axis=1, inplace=True)

In [12]:
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Missing',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}


bushfire_df['origin'] = bushfire_df['general_cause'].map(map_cause)

In [13]:
cols_to_fill = ['controlled_date', 'controlled_time']
bushfire_df[cols_to_fill] = bushfire_df[cols_to_fill].fillna('unknown')

In [14]:
output_file = './cleaned-bushfire-data.csv'
bushfire_df.to_csv(output_file, index=False, encoding='utf-8')

In [15]:
cleaned_df = pd.read_csv('./cleaned-bushfire-data.csv')

In [16]:
cleaned_df['controlled_time'] = pd.to_numeric(cleaned_df['controlled_time'], errors='coerce').fillna('unknown').astype(str)

In [17]:
# TODO: Test all of the fire_size and see if the match with their corresponding fire_class
fire_size_classification = [{"size_min": 0.1, "size_max": 0.25, "class": "A"}, {"size_min": 0.26, "size_max": 9, "class": "B"}, { "size_min": 9.1, "size_max": 99, "class": "C"}, { "size_min": 100, "size_max": 299, "class": "D"}, { "size_min": 300, "size_max": 999, "class": "E"}, { "size_min": 1000, "size_max": 4999, "class": "F"}, { "size_min": 5000, "size_max": 9999, "class": "G"}]

In [18]:
cleaned_df.drop(columns=['controlled_time', 'controlled_date'], inplace=True)

In [19]:
cleaned_df.drop(cleaned_df.loc[cleaned_df['origin'] == 'Missing'].index, inplace=True)

In [20]:
cleaned_df.drop(cleaned_df.loc[cleaned_df['origin'] == 'Accidental'].index, inplace=True)

In [21]:
cleaned_df.drop(cleaned_df.loc[cleaned_df['origin'] == 'Criminal'].index, inplace=True)

In [22]:
cleaned_df['_mean_discovery_datetime'] = pd.to_datetime(cleaned_df['discovery_datetime'], format='%Y-%m-%d %H:%M').dt.date

In [128]:
# isSpatialWithinBounds(cleaned_df['latitude'], cleaned_df['longitude'], "CA")

cleaned_df.loc[cleaned_df['longitude'] > -68]


,object_id,general_cause,fire_size,fire_class,latitude,longitude,state,discovery_day,discovery_datetime,origin,_mean_discovery_datetime
175078,175079,Natural,1.0,B,45.366700,-67.499400,ME,Sunday,1995-09-03 15:00,Natural,1995-09-03
359401,359402,Natural,0.1,A,44.866667,-67.250000,ME,Saturday,1993-08-28 18:00,Natural,1993-08-28
362899,362900,Natural,3.0,B,45.109444,-67.372222,ME,Friday,1999-07-23 13:00,Natural,1999-07-23
362900,362901,Natural,0.8,B,44.854167,-67.226667,ME,Friday,1999-07-23 17:30,Natural,1999-07-23
369348,369349,Natural,0.1,A,45.125556,-67.273056,ME,Thursday,2005-08-11 19:15,Natural,2005-08-11
...,...,...,...,...,...,...,...,...,...,...,...
2301712,2301713,Natural,0.1,A,45.550130,-67.564000,ME,Saturday,2020-06-27 00:00,Natural,2020-06-27
2301714,2301715,Natural,3.5,B,45.121720,-67.470920,ME,Saturday,2020-06-27 00:00,Natural,2020-06-27
2301717,2301718,Natural,0.1,A,45.198721,-67.968659,ME,Sunday,2020-07-05 00:00,Natural,2020-07-05
2301721,2301722,Natural,0.1,A,45.256440,-67.666280,ME,Saturday,2020-06-27 00:00,Natural,2020-06-27
